<a href="https://colab.research.google.com/github/Peeyusj/gpt_from_sratch/blob/main/gpt_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
# Cell 1 - download the dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-05-01 10:35:21--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-05-01 10:35:21 (22.8 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [30]:
# Cell 2 - read it and explore
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Total characters:", len(text))
print("First 200 characters:")
print(text[:200])

Total characters: 1115394
First 200 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [31]:
vocab= sorted(set(text))
print(len(vocab))
print(vocab)

65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [32]:
stoi={}
itos={}
for i, ch in enumerate(vocab):
    itos[i]=ch
    stoi[ch]=i

print(stoi)
print(itos)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i',

In [33]:
def encode(strVal):
  res=[]
  for i in strVal:
    res.append(stoi[i])
  return res

def decode(intVal):
  res=[]
  for i in intVal:
    res.append(itos[i])
  return "".join(res)

print(encode("hello"))
print(decode(encode("hello")))


[46, 43, 50, 50, 53]
hello


In [34]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape)
print(data[:10])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [35]:
n = int(0.9 * len(data))

train_data =data[:n]   # everything before n
val_data =data[n:]   # everything from n onwards
print('train_data-',train_data.shape)
print('val_data-',val_data.shape)

train_data- torch.Size([1003854])
val_data- torch.Size([111540])


In [36]:
block_size = 8
batch_size = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # now build x and y using ix
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [37]:
xb, yb = get_batch('train')
print(xb.shape)
print(yb.shape)
print("Input batch:")
print(xb)
print("\nTarget batch:")
print(yb)

torch.Size([4, 8])
torch.Size([4, 8])
Input batch:
tensor([[47, 57,  1, 42, 43, 57, 43, 56],
        [52, 45,  1, 44, 53, 56, 58, 59],
        [57, 12,  1, 21, 57,  1, 58, 46],
        [58,  1, 21,  1, 57, 46, 53, 59]])

Target batch:
tensor([[57,  1, 42, 43, 57, 43, 56, 60],
        [45,  1, 44, 53, 56, 58, 59, 52],
        [12,  1, 21, 57,  1, 58, 46, 47],
        [ 1, 21,  1, 57, 46, 53, 59, 50]])


In [59]:
import torch.nn as nn
from torch.nn import functional as F

n_embd = 64

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.Sequential(
            Block(n_embd, n_heads=4),
            Block(n_embd, n_heads=4),
            Block(n_embd, n_heads=4),
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        tok_emb = self.token_embedding_table(idx)
        x = self.blocks(tok_emb)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            return logits, None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [60]:
vocab_size = len(vocab)
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(loss)

tensor(4.4748, grad_fn=<NllLossBackward0>)


In [63]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for step in range(15000):
    xb, yb = get_batch('train')          # step 1: get batch
    logits, loss = model(xb, yb)         # step 2: forward pass
    optimizer.zero_grad()                # step 3: zero gradients
    loss.backward()                      # step 4: backward pass
    optimizer.step()                     # step 5: update weights

    if step % 1000 == 0:
        print(f"step {step}: loss {loss.item():.4f}")

step 0: loss 2.5872
step 1000: loss 2.3307
step 2000: loss 2.1090
step 3000: loss 2.1531
step 4000: loss 2.9068
step 5000: loss 2.5012
step 6000: loss 2.4464
step 7000: loss 2.7478
step 8000: loss 2.4218
step 9000: loss 2.3378
step 10000: loss 2.3971
step 11000: loss 2.8991
step 12000: loss 1.8634
step 13000: loss 2.2127
step 14000: loss 2.4814


In [64]:
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=200)
print(decode(generated[0].tolist()))


As glevie:
Who, fat wen beansit sonizh a otno gomnen to cnroven seorde onord orear
O wullls'ancs
I d, hand oun as taror adealint nor Gathaord I wwe'm, Lthant heremis; waullll?

CPaiy upe'encid allls a


In [52]:
x = torch.tensor([
    [1.0, 2.0],   # token 0
    [3.0, 4.0],   # token 1
    [5.0, 6.0],   # token 2
    [7.0, 8.0],   # token 3
])

In [ ]:
T, C = x.shape

x_avg = torch.zeros((T, C))

for t in range(T):
    x_prev = x[:t+1]
    x_avg[t] = x_prev.mean(dim=0)

print(x_avg)

In [ ]:
torch.tril(torch.ones(T, T))

In [ ]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(dim=1, keepdim=True)
print(wei)

In [ ]:
x_avg2 = wei @ x
print(x_avg2)

In [ ]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
print(wei)

In [ ]:
wei = F.softmax(wei, dim=-1)
print(wei)

In [ ]:
out = wei @ x
print(out)

In [ ]:
torch.manual_seed(1337)
head_size = 16

key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)

wei = q @ k.transpose(-2, -1)

print(k.shape)
print(q.shape)
print(wei.shape)
print(wei)

In [ ]:
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
print(wei)

In [ ]:
wei = q @ k.transpose(-2, -1) * head_size**-0.5
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
print(wei)

In [ ]:
v = value(x)      # transform each token through value layer
out = wei @ v     # weighted mix of value vectors
print(out.shape)

In [45]:
class Head(nn.Module):
    def __init__(self, C, head_size):
        super().__init__()
        self.key = nn.Linear(C, head_size, bias=False)
        self.query = nn.Linear(C, head_size, bias=False)
        self.value = nn.Linear(C, head_size, bias=False)
        self.head_size = head_size
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * self.head_size**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)
        out = wei @ v
        return out

In [46]:
import inspect
print(inspect.signature(Head.__init__))

(self, C, head_size)


In [47]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        # "store num_heads Head objects in a list"
        self.heads = nn.ModuleList([Head(n_embd, head_size) for _ in range(num_heads)])

    def forward(self, x):
        # "run x through each head, combine outputs"
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [48]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # expand
            nn.ReLU(),                        # non-linearity
            nn.Linear(4 * n_embd, n_embd),   # compress back
        )

    def forward(self, x):
        return self.net(x)

In [49]:
class Block(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x